# Import thư viện

In [34]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')
from sklearn.metrics import precision_score, recall_score, roc_curve, classification_report, confusion_matrix

# Tổng hợp dữ liệu thô đã thu thập vào 1 file

In [23]:
FEATURE_MAP = {
    'Date': 'Date',
    'Time': 'Time',
    'Total CPU Usage [%]':  'CPU_Load_Pct',
    'CPU Package [°C]': 'Temperature_C',
    'Other (Docking) [RPM]': 'Fan_RPM',
    'Core VIDs (avg) [V]': 'Voltage_V',
}
FILES = [
    ('Normal.CSV',   0),   # 0: Bình thường
    ('Warning.csv',  1),   # 1: Cảnh báo
    ('Critical.CSV', 2),   # 2: Nguy hiểm
]

dfs = []

for path, label in FILES:
    df = pd.read_csv(path, encoding='latin-1', usecols=list(FEATURE_MAP.keys()))
    df = df.rename(columns=FEATURE_MAP)
    df['Label'] = label
    print(f"[Label={label}] {path.split('/')[-1]} → {len(df)} dòng")
    dfs.append(df)

# Gộp + trộn ngẫu nhiên
merged = pd.concat(dfs, ignore_index=True)
merged = merged.sample(frac=1, random_state=42).reset_index(drop=True)

# Lưu ra file
out = 'fan_dataset.csv'
merged.to_csv(out, index=False)

# Tóm tắt
print(f"\nTổng: {len(merged)} dòng  →  lưu vào {out}")
print(merged['Label'].value_counts().rename({0: 'Normal', 1: 'Warning', 2: 'Critical'}).to_string())
print("\nMẫu đầu tiên:")
print(merged.head(5).to_string(index=False))

[Label=0] Normal.CSV → 1188 dòng
[Label=1] Warning.csv → 659 dòng
[Label=2] Critical.CSV → 324 dòng

Tổng: 2171 dòng  →  lưu vào fan_dataset.csv
Label
Normal      1188
Warning      659
Critical     324

Mẫu đầu tiên:
    Date    Time  Voltage_V  CPU_Load_Pct  Temperature_C  Fan_RPM  Label
1.7.2026 54:00.5      0.731          22.3             50     7308      1
1.7.2026 02:29.5      0.876          22.1             55     4801      0
1.7.2026 43:37.6      0.727          16.8             47     7295      1
1.7.2026 16:38.0      0.842          16.4             49     4789      0
1.7.2026 19:58.7      0.658          27.5             54     4812      0


# Đọc dữ liệu

In [24]:
df = pd.read_csv('fan_dataset.csv')

FEATURES = ['CPU_Load_Pct', 'Temperature_C', 'Fan_RPM', 'Voltage_V']
TARGET = 'Label'

In [45]:
def remove_outliers_iqr_per_class(df, cols, label_col):
    dfs = []
    for label, group in df.groupby(label_col):
        mask = pd.Series([True] * len(group), index=group.index)
        for col in cols:
            Q1 = group[col].quantile(0.25)
            Q3 = group[col].quantile(0.75)
            IQR = Q3 - Q1
            lower = Q1 - 1.5 * IQR
            upper = Q3 + 1.5 * IQR
            mask &= group[col].between(lower, upper)
        dfs.append(group[mask])
    return pd.concat(dfs).reset_index(drop=True)

df_clean = remove_outliers_iqr_per_class(df, FEATURES, TARGET)
print(f"[Outlier] {len(df)} → {len(df_clean)} dòng (loại {len(df)-len(df_clean)} outlier)")
print(df_clean[TARGET].value_counts().rename({0:'Normal',1:'Warning',2:'Critical'}))

[Outlier] 2171 → 1929 dòng (loại 242 outlier)
Label
Normal      1140
Warning      483
Critical     306
Name: count, dtype: int64


# Rút trích đặc trưng

In [46]:
# Tính thêm các đặc trưng thống kê trên sliding window
W = 10     # cửa sổ trích đặc trưng

def extract_features_window(series, prefix):
    # Tính mean, std, min, max, range trên sliding window 
    feats = {}
    feats[f'{prefix}_mean'] = series.rolling(W, min_periods=1).mean()
    feats[f'{prefix}_std'] = series.rolling(W, min_periods=1).std().fillna(0)
    feats[f'{prefix}_min'] = series.rolling(W, min_periods=1).min()
    feats[f'{prefix}_max'] = series.rolling(W, min_periods=1).max()
    feats[f'{prefix}_range'] = feats[f'{prefix}_max'] - feats[f'{prefix}_min']
    return pd.DataFrame(feats)

list_feat_dfs = [df[FEATURES + [TARGET]]]

for col in FEATURES:
    df_window = extract_features_window(df[col], col)
    list_feat_dfs.append(df_window)

df_feat = pd.concat(list_feat_dfs, axis=1)

# Thêm tỉ lệ tốc độ quạt / nhiệt độ (Fan efficiency proxy)
# Khi quạt hỏng: RPM thấp nhưng nhiệt độ cao → tỉ lệ này giảm mạnh
df_feat['Fan_Temp_Ratio'] = (
    df_feat['Fan_RPM'] / (df_feat['Temperature_C'] + 1e-6)
)
df_feat = df_feat.dropna().reset_index(drop=True)
print(f"[Feature] Tổng số feature sau rút trích: {len([c for c in df_feat.columns if c != TARGET])}")
print(f"Gồm: 4 raw + 5×4 window stats + 1 ratio = {4 + 5*4 + 1} features")

[Feature] Tổng số feature sau rút trích: 25
Gồm: 4 raw + 5×4 window stats + 1 ratio = 25 features


# Kiểm tra & Lưu

In [41]:
print(f"\n[Preview] Shape: {df_feat.shape}")
print(df_feat[['CPU_Load_Pct','Temperature_C','Fan_RPM','Voltage_V',
        'CPU_Load_Pct_mean','Temperature_C_std','Fan_Temp_Ratio', TARGET]].head(8).round(4).to_string(index=False))
 
df_feat.to_csv('fan_dataset_features.csv', index=False)
print(f"\n[Done] Lưu vào fan_dataset_features.csv")
print(f"       Phân phối label:\n{df_feat[TARGET].value_counts().rename({0:'Normal',1:'Warning',2:'Critical'}).to_string()}")


[Preview] Shape: (2171, 26)
 CPU_Load_Pct  Temperature_C  Fan_RPM  Voltage_V  CPU_Load_Pct_mean  Temperature_C_std  Fan_Temp_Ratio  Label
         22.3             50     7308      0.731            22.3000             0.0000        146.1600      1
         22.1             55     4801      0.876            22.2000             3.5355         87.2909      0
         16.8             47     7295      0.727            20.4000             4.0415        155.2128      1
         16.4             49     4789      0.842            19.4000             3.4034         97.7347      0
         27.5             54     4812      0.658            21.0200             3.3912         89.1111      0
         17.9             54     5028      0.752            20.5000             3.2711         93.1111      0
         20.4             57     4812      0.996            20.4857             3.6384         84.4211      0
        100.0             90     7322      0.864            30.4250            13.7529     

# Phân chia tập train/test

In [42]:
X = df_feat.drop(columns=[TARGET])
y = df_feat[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, 
    test_size=0.2,
    shuffle=False
)
print(f"Kích thước tập train: {X_train.shape}")
print(f"Kích thước tập test: {X_test.shape}")
scaler = MinMaxScaler()

# Chỉ FIT (học max/min) trên tập Train, sau đó transform
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns)

# Dùng max/min của Train để transform Test (Không được fit lại trên Test)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns)

print(f"Kích thước tập train: {X_train_scaled.shape}")
print(f"Kích thước tập test: {X_test_scaled.shape}")

Kích thước tập train: (1736, 25)
Kích thước tập test: (435, 25)
Kích thước tập train: (1736, 25)
Kích thước tập test: (435, 25)


# Random Forest Classifier

In [ ]:
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV

rf_model = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)
rf_model.fit(X_train_scaled, y_train)
y_pred_rf = rf_model.predict(X_test_scaled)

print(f"Precision: {precision_score(y_test, y_pred_rf, average='macro'):.4f}")
print(f"Recall: {recall_score(y_test, y_pred_rf, average='macro'):.4f}")
print(classification_report(y_test, y_pred_rf, target_names=['Normal','Warning','Critical']))
cm = confusion_matrix(y_test, y_pred_rf)
print("Confusion matrix Random Forest:")
print(pd.DataFrame(cm, index=['Normal','Warning','Critical'], columns=['Normal','Warning','Critical']))

# Xem feature nào ảnh hưởng nhiều nhất đến quyết định phân loại
importances = pd.Series(rf_model.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop 10 feature quan trọng nhất:")
print(importances.head(10).round(4).to_string())

Precision: 0.9700
Recall: 0.9498
              precision    recall  f1-score   support

      Normal       0.92      1.00      0.96       229
     Warning       1.00      0.85      0.92       130
    Critical       0.99      1.00      0.99        76

    accuracy                           0.95       435
   macro avg       0.97      0.95      0.96       435
weighted avg       0.96      0.95      0.95       435

Confusion matrix Random Forest:
          Normal  Warning  Critical
Normal       228        0         1
Warning       19      111         0
Critical       0        0        76

Top 10 feature quan trọng nhất:
CPU_Load_Pct          0.3172
Fan_RPM               0.2580
Fan_Temp_Ratio        0.1654
Temperature_C         0.1064
Voltage_V             0.0689
CPU_Load_Pct_std      0.0165
CPU_Load_Pct_max      0.0137
CPU_Load_Pct_mean     0.0091
CPU_Load_Pct_range    0.0077
Fan_RPM_mean          0.0074


# Isolation Forest (Anomaly Detection)

In [52]:
# Đổi nhãn gốc sang nhãn Anomaly
# Normal (0) -> 1 (Inlier)
# Warning (1) & Critical (2) -> -1 (Outlier)
y_test_anomaly = y_test.apply(lambda x: 1 if x == 0 else -1)

# Khởi tạo và Huấn luyện Isolation Forest
# Tính tỷ lệ lỗi thực tế trong tập Train để báo cho model biết (Contamination)
outlier_ratio = sum(y_train != 0) / len(y_train)

iso_model = IsolationForest(
    n_estimators=500, 
    contamination=outlier_ratio,  # Tỷ lệ bất thường kỳ vọng
    random_state=42,
    n_jobs=-1
)
iso_model.fit(X_train_scaled)

# Dự đoán trên tập Test
y_pred_iso = iso_model.predict(X_test_scaled)

# Đánh giá kết quả
print("=== KẾT QUẢ ISOLATION FOREST ===")
print("Quy ước:  1 = Bình thường (Normal)  |  -1 = Bất thường (Warning + Critical)\n")
print(classification_report(y_test_anomaly, y_pred_iso, target_names=['Bất thường', 'Bình thường']))

# Vẽ Confusion Matrix
cm_iso = confusion_matrix(y_test_anomaly, y_pred_iso)
cm_df = pd.DataFrame(
    cm_iso, 
    index=['Bất thường', 'Bình thường'], 
    columns=['Bất thường', 'Bình thường']
)
print("Confusion Matrix Isolation Forest:")
print(cm_df)

=== KẾT QUẢ ISOLATION FOREST ===
Quy ước:  1 = Bình thường (Normal)  |  -1 = Bất thường (Warning + Critical)

              precision    recall  f1-score   support

  Bất thường       0.62      0.63      0.63       206
 Bình thường       0.66      0.66      0.66       229

    accuracy                           0.64       435
   macro avg       0.64      0.64      0.64       435
weighted avg       0.64      0.64      0.64       435

Confusion Matrix Isolation Forest:
             Bất thường  Bình thường
Bất thường          130           76
Bình thường          79          150


# Logistic Regression (Risk Calibration)

In [55]:
from sklearn.calibration import CalibratedClassifierCV

rf_base = RandomForestClassifier(
    n_estimators=500,
    max_depth=5,
    min_samples_leaf=10,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
)

# Dùng Logistic Regression để hiệu chuẩn rủi ro
calibrated_rf = CalibratedClassifierCV(
    estimator=rf_base,
    method='sigmoid',
    cv=3
)

# Huấn luyện mô hình đã hiệu chuẩn
calibrated_rf.fit(X_train_scaled, y_train)

# Dự đoán xác suất thực trên tập test
probs = calibrated_rf.predict_proba(X_test_scaled)

# Định nghĩa chỉ số rủi ro (Risk Index từ 0-100%)
# Công thức tính Risk Index dựa trên trọng số nguy hiểm:
# Risk = %Warning * 0.4 + %Critical * 1.0
risk_index = (probs[:, 1] * 40 + probs[:, 2] * 100)

# Gộp kết quả vào DataFrame để quan sát trực quan chuỗi thời gian
calibration_results = pd.DataFrame({
    'Thực tế (Label)': y_test.values,
    'P_Normal (%)': (probs[:, 0] * 100).round(2),
    'P_Warning (%)': (probs[:, 1] * 100).round(2),
    'P_Critical (%)': (probs[:, 2] * 100).round(2),
    'Risk_Index (%)': risk_index.round(2)
})

# ==========================================
# IN KẾT QUẢ ĐỂ KIỂM TRA
# ==========================================
print("=== KẾT QUẢ HIỆU CHUẨN RỦI RO (RISK CALIBRATION) ===")
print("\n[Mẫu Dữ Liệu Thực Tế Theo Thời Gian Sau Hiệu Chuẩn]:")
# Xem thử 15 dòng ngẫu nhiên hoặc liên tục để thấy Risk Index tăng/giảm mượt mà
print(calibration_results.sample(15, random_state=10).to_string(index=False))

# Đánh giá xem Risk Index trung bình của từng nhóm có phân cấp đẹp không
print("\n[Risk Index Trung Bình Theo Từng Trạng Thái Thực Tế]:")
print(calibration_results.groupby('Thực tế (Label)')['Risk_Index (%)'].mean().rename({0:'Normal', 1:'Warning', 2:'Critical'}))

=== KẾT QUẢ HIỆU CHUẨN RỦI RO (RISK CALIBRATION) ===

[Mẫu Dữ Liệu Thực Tế Theo Thời Gian Sau Hiệu Chuẩn]:
 Thực tế (Label)  P_Normal (%)  P_Warning (%)  P_Critical (%)  Risk_Index (%)
               1          0.33          98.94            0.73           40.31
               1          0.26          99.58            0.16           39.99
               0         97.37           2.50            0.13            1.13
               0         93.92           5.95            0.13            2.51
               0         98.39           1.48            0.13            0.72
               2          0.16           0.89           98.95           99.31
               1          0.90          98.96            0.14           39.73
               0         95.13           4.71            0.16            2.04
               0         92.25           7.61            0.14            3.18
               0         97.32           2.45            0.22            1.21
               1          0.39     